# 02 — Chunking Analysis

**Purpose:** Analyse the JSONL chunks produced by `scripts/build_corpus.py` to understand
chunk quality before building any retrieval index.

**Sections**
0. Setup  
1. Load chunks  
2. Corpus overview (documents, chunks, tokens)  
3. Token-length statistics (min/max/mean/median/std/P25/P75)  
4. Distribution of chunk lengths (histogram + strip plot)  
5. Distribution grouped by strategy  
6. Chunks per document  
7. Very short chunks  
8. Unusually long chunks  
9. Section / heading coverage  
10. Qualitative examples (good / too-short / too-long / stranded-heading)  
11. Summary health dashboard  
12. **Recommendation** — freeze or revise?

Uses `src.preprocessing.chunker.load_chunks()` — no logic duplicated.

---
**If the corpus does not exist yet**, build it first:
```bash
python scripts/build_corpus.py \
    --raw-dir data/raw --strategy sentence --chunk-size 256 --overlap 32
```

## 0. Setup

In [ ]:
import os, re, sys, textwrap, warnings
from pathlib import Path

# ------------------------------------------------------------------
# Set Agg backend BEFORE importing pyplot so the notebook executes
# cleanly in headless / nbconvert environments.  When run inside
# JupyterLab the inline backend takes over automatically.
# ------------------------------------------------------------------
import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')

# ── project root = one level above notebooks/ ─────────────────────
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.chunker import load_chunks

# ── paths ──────────────────────────────────────────────────────────
CHUNKS_FILE = PROJECT_ROOT / 'data' / 'chunks' / 'chunks.jsonl'
RAW_DIR     = PROJECT_ROOT / 'data' / 'raw'
FIG_DIR     = PROJECT_ROOT / 'notebooks'

# ── style ──────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
BLUE    = '#2563EB'
RED     = '#EF4444'
GREEN   = '#16A34A'
AMBER   = '#F59E0B'
PURPLE  = '#8B5CF6'

# ── thresholds (edit here; referenced throughout) ─────────────────
SHORT_THR = 20    # tokens  – below = "too short"
LONG_THR  = 300   # tokens  – above = "too long"

print(f'Project root : {PROJECT_ROOT}')
print(f'Chunks file  : {CHUNKS_FILE}')
print(f'matplotlib   : {matplotlib.__version__}')
print(f'Short / long thresholds : {SHORT_THR} / {LONG_THR} tokens')

## 1. Load Chunks

In [ ]:
# ── guard: corpus absent or empty ─────────────────────────────────
if not CHUNKS_FILE.exists() or CHUNKS_FILE.stat().st_size == 0:
    msg = [
        '=' * 62,
        '  CORPUS NOT FOUND',
        '=' * 62,
        f'Expected : {CHUNKS_FILE}',
        '',
        'Build the corpus first:',
        '',
        '  python scripts/build_corpus.py \\',
        '      --raw-dir    data/raw       \\',
        '      --proc-dir   data/processed \\',
        '      --chunk-dir  data/chunks    \\',
        '      --strategy   sentence       \\',
        '      --chunk-size 256            \\',
        '      --overlap    32',
        '',
        f'Place PDF / HTML / TXT / MD files in: {RAW_DIR}',
    ]
    print('\n'.join(msg))
    raise SystemExit('Notebook halted — no corpus data.')

raw_chunks = load_chunks(CHUNKS_FILE)
print(f'Loaded {len(raw_chunks)} chunks  from  {CHUNKS_FILE}')

In [ ]:
# ── build analysis DataFrame ───────────────────────────────────────
rows = []
for c in raw_chunks:
    rows.append({
        'chunk_id'    : c.chunk_id,
        'doc_id'      : c.doc_id,
        'chunk_index' : c.metadata.get('chunk_index', 0),
        'total_chunks': c.metadata.get('total_chunks', 0),
        'tokens'      : len(c.text.split()),
        'chars'       : len(c.text),
        'strategy'    : c.metadata.get('strategy', 'unknown'),
        'title'       : c.metadata.get('title', ''),
        'section'     : c.metadata.get('section', ''),
        'source'      : Path(c.metadata.get('source', '')).name,
        'start_char'  : c.start_char,
        'end_char'    : c.end_char,
        'text'        : c.text,
    })

df = pd.DataFrame(rows)
df['is_short']     = df['tokens'] < SHORT_THR
df['is_long']      = df['tokens'] > LONG_THR
df['has_section']  = df['section'].str.strip().astype(bool)
df['starts_head']  = df['text'].str.lstrip().str.startswith('#')

print(f'Shape: {df.shape}')
df[['chunk_id', 'tokens', 'strategy', 'section', 'source']].head(5)

## 2. Corpus Overview

In [ ]:
n_docs       = df['doc_id'].nunique()
n_chunks     = len(df)
strategies   = df['strategy'].unique().tolist()
total_tokens = int(df['tokens'].sum())

print('\u2550' * 54)
print('  CORPUS OVERVIEW')
print('\u2550' * 54)
print(f'  Total documents          : {n_docs}')
print(f'  Total chunks             : {n_chunks}')
print(f'  Total tokens (approx.)   : {total_tokens:,}')
print(f'  Chunking strategy/ies    : {", ".join(strategies)}')
print(f'  JSONL file size          : {CHUNKS_FILE.stat().st_size / 1024:.1f} KB')
print()

per_doc = (
    df.groupby('doc_id')
      .agg(
          source       = ('source',   'first'),
          n_chunks     = ('chunk_id', 'count'),
          total_tokens = ('tokens',   'sum'),
          min_tok      = ('tokens',   'min'),
          max_tok      = ('tokens',   'max'),
          mean_tok     = ('tokens',   'mean'),
      )
      .reset_index()
)
per_doc['mean_tok'] = per_doc['mean_tok'].round(1)

print('Per-document breakdown:')
print(per_doc[['source','n_chunks','total_tokens',
               'min_tok','max_tok','mean_tok']].to_string(index=False))

## 3. Token-Length Statistics

In [ ]:
tl = df['tokens']

stats = {
    'Count'  : int(tl.count()),
    'Min'    : int(tl.min()),
    'P25'    : float(tl.quantile(0.25)),
    'Median' : float(tl.median()),
    'Mean'   : round(float(tl.mean()), 2),
    'P75'    : float(tl.quantile(0.75)),
    'Max'    : int(tl.max()),
    'Std Dev': round(float(tl.std()), 2),
}

print('Token-Length Statistics')
print('\u2500' * 34)
for k, v in stats.items():
    print(f'  {k:<10} : {v}')
print('\u2500' * 34)

## 4. Distribution of Chunk Lengths

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Chunk Token-Length Distribution',
             fontsize=14, fontweight='bold', y=1.01)

# ── left: histogram ───────────────────────────────────────────────
ax = axes[0]
n_bins = max(8, min(30, n_chunks // 2))
ax.hist(df['tokens'], bins=n_bins, color=BLUE, edgecolor='white', lw=0.6)
ax.axvline(stats['Mean'],   color='#F97316', lw=1.8, ls='--',
           label=f'Mean ({stats["Mean"]:.0f})')
ax.axvline(stats['Median'], color=GREEN, lw=1.8, ls=':',
           label=f'Median ({stats["Median"]:.0f})')
ax.axvspan(0, SHORT_THR,
           alpha=0.10, color=RED,   label=f'Short < {SHORT_THR}')
ax.axvspan(LONG_THR, df['tokens'].max() + 20,
           alpha=0.10, color=AMBER, label=f'Long > {LONG_THR}')
ax.set_xlabel('Token count')
ax.set_ylabel('Number of chunks')
ax.set_title('Histogram')
ax.legend(fontsize=9)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# ── right: strip plot per strategy ────────────────────────────────
# (pure scatter + hlines avoids the matplotlib boxplot/tick-copy bug
#  present in matplotlib < 3.11 on Python 3.14)
ax2 = axes[1]
strats = sorted(df['strategy'].unique())
pal    = sns.color_palette('muted', len(strats))

for i, (s, col) in enumerate(zip(strats, pal)):
    vals = df.loc[df['strategy'] == s, 'tokens'].values
    rng  = np.random.default_rng(i)
    jit  = rng.uniform(-0.18, 0.18, size=len(vals))
    ax2.scatter(np.full(len(vals), i) + jit, vals,
                color=col, s=55, alpha=0.78, label=s, zorder=3)
    ax2.hlines(np.median(vals), i - 0.28, i + 0.28,
               colors='#0F172A', lw=2.4, zorder=4)   # median marker

ax2.set_xticks(range(len(strats)))
ax2.set_xticklabels(strats)
ax2.set_xlabel('Strategy')
ax2.set_ylabel('Token count')
ax2.set_title('Strip plot by Strategy  (bar = median)')
ax2.axhline(SHORT_THR, color=RED,   lw=1, ls='--', alpha=0.55,
            label=f'Short ({SHORT_THR})')
ax2.axhline(LONG_THR,  color=AMBER, lw=1, ls='--', alpha=0.55,
            label=f'Long ({LONG_THR})')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / '_fig04_token_dist.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Figure saved.')

## 5. Distribution Grouped by Strategy

In [ ]:
# ── stats table ───────────────────────────────────────────────────
strat_stats = (
    df.groupby('strategy')['tokens']
      .agg(
          N      = 'count',
          Min    = 'min',
          P25    = lambda x: round(x.quantile(0.25), 1),
          Median = lambda x: round(x.median(), 1),
          Mean   = lambda x: round(x.mean(), 1),
          P75    = lambda x: round(x.quantile(0.75), 1),
          Max    = 'max',
          Std    = lambda x: round(x.std(), 1),
      )
)
print('Token-length statistics by strategy:')
print(strat_stats.to_string())
print()

# ── KDE for large corpus, strip for small ─────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))

if n_chunks >= 20:
    for s, grp in df.groupby('strategy'):
        grp['tokens'].plot.kde(ax=ax, label=s, lw=2)
    ax.axvline(SHORT_THR, color=RED,   lw=1, ls='--', alpha=0.5,
               label=f'Short ({SHORT_THR})')
    ax.axvline(LONG_THR,  color=AMBER, lw=1, ls='--', alpha=0.5,
               label=f'Long ({LONG_THR})')
    ax.set_xlabel('Token count')
    ax.set_ylabel('Density')
    ax.set_title('Token-length KDE by Strategy')
else:
    cols = sns.color_palette('muted', df['strategy'].nunique())
    for i, (s, grp) in enumerate(df.groupby('strategy')):
        jit = np.random.default_rng(i).uniform(-0.15, 0.15, len(grp))
        ax.scatter(np.full(len(grp), i) + jit, grp['tokens'],
                   label=s, s=60, alpha=0.8, color=cols[i])
    ax.set_xticks(range(df['strategy'].nunique()))
    ax.set_xticklabels(sorted(df['strategy'].unique()))
    ax.set_xlabel('Strategy')
    ax.set_ylabel('Token count')
    ax.set_title('Tokens by Strategy  (small corpus — strip)')

ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / '_fig05_strategy_kde.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close(fig)

## 6. Chunks per Document

In [ ]:
cpd = per_doc.set_index('source')['n_chunks'].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(max(6, len(cpd) * 2.0), 4))
bars = ax.bar(range(len(cpd)), cpd.values, color=BLUE, edgecolor='white')
ax.set_xticks(range(len(cpd)))
ax.set_xticklabels(
    [textwrap.fill(s, 20) for s in cpd.index],
    rotation=15, ha='right', fontsize=9,
)
ax.set_ylabel('Number of chunks')
ax.set_title('Chunks per Document')
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
for bar, val in zip(bars, cpd.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.05, str(val),
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(FIG_DIR / '_fig06_chunks_per_doc.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close(fig)

print(f'Min: {cpd.min()}  |  Max: {cpd.max()}  |  Mean: {cpd.mean():.1f} chunks/doc')

## 7. Very Short Chunks

In [ ]:
short_df  = df[df['is_short']].copy()
n_short   = len(short_df)
pct_short = 100 * n_short / n_chunks

print(f'Very short chunks  (< {SHORT_THR} tokens) : '
      f'{n_short} / {n_chunks}  ({pct_short:.1f}%)')
print()

if n_short == 0:
    print('  \u2713  None found.')
else:
    print(f'  \u26a0  {n_short} short chunk(s) detected:')
    print(short_df[['chunk_id','tokens','section','source']].to_string(index=False))
    print()
    for _, row in short_df.iterrows():
        print(f'  [{row["chunk_id"]}]  ({row["tokens"]} tokens)')
        print(f'  Text: {repr(row["text"][:300])}')
        print()

## 8. Unusually Long Chunks

In [ ]:
long_df  = df[df['is_long']].copy()
n_long   = len(long_df)
pct_long = 100 * n_long / n_chunks

print(f'Unusually long chunks  (> {LONG_THR} tokens) : '
      f'{n_long} / {n_chunks}  ({pct_long:.1f}%)')
print()

if n_long == 0:
    print('  \u2713  None found.')
else:
    print(f'  \u26a0  {n_long} long chunk(s) detected:')
    print(long_df[['chunk_id','tokens','section','source']].to_string(index=False))
    print()
    for _, row in long_df.iterrows():
        print(f'  [{row["chunk_id"]}]  ({row["tokens"]} tokens)')
        preview = ' '.join(row['text'].split()[:40])
        print(f'  First 40 tokens: {preview}...')
        print()

## 9. Section / Heading Coverage

In [ ]:
n_with     = int(df['has_section'].sum())
n_without  = n_chunks - n_with
pct_with   = 100 * n_with / n_chunks

print(f'Has section label  : {n_with} / {n_chunks}  ({pct_with:.1f}%)')
print(f'No section label   : {n_without} / {n_chunks}  ({100 - pct_with:.1f}%)')
print()

no_sec       = df[~df['has_section']]
first_chunks = no_sec[no_sec['chunk_index'] == 0]
mid_chunks   = no_sec[no_sec['chunk_index'] >  0]

print(f'  Of the {n_without} sectionless chunks:')
print(f'    First of document (expected)  : {len(first_chunks)}')
print(f'    Mid-document (potential issue): {len(mid_chunks)}')

if len(mid_chunks):
    print()
    print('  Mid-document sectionless chunks:')
    print(mid_chunks[['chunk_id','chunk_index','title','source']].to_string(index=False))
print()

sec_freq = (
    df[df['has_section']]
      .groupby('section')['chunk_id'].count()
      .sort_values(ascending=False)
      .reset_index()
      .rename(columns={'chunk_id': 'n_chunks'})
)
print('Section label frequency (top 15):')
print(sec_freq.head(15).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Section / Heading Coverage', fontsize=14, fontweight='bold')

# pie
ax = axes[0]
ax.pie(
    [n_with, n_without],
    labels=[f'Has section\n({n_with})', f'No section\n({n_without})'],
    colors=[GREEN, '#CBD5E1'],
    autopct='%1.1f%%',
    wedgeprops={'linewidth': 1.5, 'edgecolor': 'white'},
    startangle=90,
)
ax.set_title('Section label presence')

# horizontal bar: top sections
ax2 = axes[1]
if len(sec_freq):
    top = sec_freq.head(12)
    ax2.barh(range(len(top)), top['n_chunks'], color=BLUE, edgecolor='white')
    ax2.set_yticks(range(len(top)))
    ax2.set_yticklabels([textwrap.fill(s, 28) for s in top['section']], fontsize=8)
    ax2.invert_yaxis()
    ax2.set_xlabel('Number of chunks')
    ax2.set_title('Chunks per Section (top 12)')
    ax2.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
else:
    ax2.text(0.5, 0.5, 'No section labels found',
             ha='center', va='center', transform=ax2.transAxes,
             fontsize=12, color='gray')

plt.tight_layout()
plt.savefig(FIG_DIR / '_fig09_sections.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close(fig)

## 10. Qualitative Examples

In [ ]:
SEP = '\u2500' * 70

def _show(label: str, row: pd.Series, max_tok: int = 80) -> None:
    toks    = row['text'].split()
    preview = ' '.join(toks[:max_tok])
    if len(toks) > max_tok:
        preview += f'  [\u2026 {len(toks)-max_tok} more tokens]'
    print(SEP)
    print(f'  {label}')
    print(f'  chunk_id : {row["chunk_id"]}   tokens={row["tokens"]}')
    print(f'  section  : {repr(row["section"])}')
    print(f'  source   : {row["source"]}')
    print(SEP)
    for line in textwrap.wrap(preview, 86):
        print('  ' + line)
    print()

### 10a. Good Chunks  
Well-sized (P25–P75 token range), has section label, coherent content.

In [ ]:
good_mask = (
    df['has_section']
    & df['tokens'].between(stats['P25'], stats['P75'])
    & ~df['is_short'] & ~df['is_long']
)
good_df = df[good_mask] if good_mask.sum() else df[~df['is_short'] & ~df['is_long']]

n_show = min(3, len(good_df))
sample = good_df.sample(n=n_show, random_state=42) if len(good_df) >= n_show else good_df

print(f'Showing {n_show} good chunk(s):\n')
for _, row in sample.iterrows():
    _show('GOOD', row)

### 10b. Too-Short Chunks

In [ ]:
if n_short == 0:
    print('\u2713  No too-short chunks.')
else:
    for _, row in short_df.iterrows():
        _show('TOO SHORT', row)

### 10c. Overly Large Chunks

In [ ]:
if n_long == 0:
    print('\u2713  No overly large chunks.')
else:
    for _, row in long_df.iterrows():
        _show('TOO LONG', row)

### 10d. Heading / Content Separation Issues  
A *stranded-heading* chunk contains a Markdown heading but fewer than 8 tokens of body text — the heading was split from its content paragraph.

In [ ]:
def _is_stranded(row: pd.Series) -> bool:
    if row['tokens'] > 15:
        return False
    lines    = [l.strip() for l in row['text'].splitlines() if l.strip()]
    headings = [l for l in lines if l.startswith('#')]
    body     = [l for l in lines if not l.startswith('#')]
    return bool(headings) and (
        not body or len(' '.join(body).split()) < 8
    )

df['stranded'] = df.apply(_is_stranded, axis=1)
stranded_df    = df[df['stranded']]
n_stranded     = len(stranded_df)

print(f'Stranded-heading chunks: {n_stranded}\n')
if n_stranded == 0:
    print('  \u2713  No heading\u2013content separation issues.')
else:
    for _, row in stranded_df.iterrows():
        _show('HEADING STRANDED', row)

## 11. Summary Health Dashboard

In [ ]:
def _health(row):
    if row['is_short']:   return 'short'
    if row['is_long']:    return 'long'
    if row['stranded']:   return 'stranded'
    return 'ok'

df['health'] = df.apply(_health, axis=1)
hc = df['health'].value_counts()

n_ok        = int(hc.get('ok',       0))
n_short_h   = int(hc.get('short',    0))
n_long_h    = int(hc.get('long',     0))
n_strand_h  = int(hc.get('stranded', 0))

print('\u2550' * 54)
print('  CHUNK HEALTH SUMMARY')
print('\u2550' * 54)
print(f'  OK (well-formed)          : {n_ok:>4}  ({100*n_ok/n_chunks:.1f}%)')
print(f'  Too short (< {SHORT_THR} tok)    : {n_short_h:>4}  ({100*n_short_h/n_chunks:.1f}%)')
print(f'  Too long  (> {LONG_THR} tok)  : {n_long_h:>4}  ({100*n_long_h/n_chunks:.1f}%)')
print(f'  Stranded heading          : {n_strand_h:>4}  ({100*n_strand_h/n_chunks:.1f}%)')
print('\u2500' * 54)
print(f'  Total                     : {n_chunks:>4}')
print('\u2550' * 54)

# ── pie ───────────────────────────────────────────────────────────
cmap   = {'ok': GREEN, 'short': RED, 'long': AMBER, 'stranded': PURPLE}
lmap   = {'ok': f'OK ({n_ok})', 'short': f'Short ({n_short_h})',
          'long': f'Long ({n_long_h})', 'stranded': f'Stranded ({n_strand_h})'}
keys   = [k for k in ['ok','short','long','stranded'] if hc.get(k, 0) > 0]

fig, ax = plt.subplots(figsize=(6, 4))
if keys:
    ax.pie(
        [hc[k] for k in keys],
        labels=[lmap[k] for k in keys],
        colors=[cmap[k] for k in keys],
        autopct='%1.1f%%',
        wedgeprops={'linewidth': 1.5, 'edgecolor': 'white'},
        startangle=90,
    )
ax.set_title('Chunk Health', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / '_fig11_health.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close(fig)

## 12. Recommendation

In [ ]:
issues    = []
positives = []

# 1. corpus size
if n_docs < 5:
    issues.append(
        f'\u26a0  CORPUS TOO SMALL: {n_docs} document(s). '
        'Retrieval metrics will not be statistically meaningful. '
        'Populate data/raw/ with real cardiology literature '
        '(guidelines, case series, systematic reviews) before '
        'running retrieval experiments.'
    )
else:
    positives.append(f'\u2713  Corpus: {n_docs} docs, {n_chunks} chunks, {total_tokens:,} tokens.')

# 2. short chunks
if pct_short > 10:
    issues.append(
        f'\u26a0  HIGH SHORT-CHUNK RATE: {pct_short:.1f}% below {SHORT_THR} tokens. '
        'Short chunks degrade all retrieval methods. '
        'Increase --chunk-size or reduce --overlap.'
    )
elif pct_short > 0:
    issues.append(
        f'\u26a0  MINOR: {n_short_h} short chunk(s) ({pct_short:.1f}%). '
        'Inspect — may be section titles; acceptable if < 5 %.'
    )
else:
    positives.append(f'\u2713  No very short chunks (< {SHORT_THR} tokens).')

# 3. long chunks
if pct_long > 10:
    issues.append(
        f'\u26a0  HIGH LONG-CHUNK RATE: {pct_long:.1f}% above {LONG_THR} tokens. '
        'Long chunks dilute dense retrieval precision and may exceed '
        'LLM context windows. Decrease --chunk-size.'
    )
elif pct_long > 0:
    issues.append(
        f'\u26a0  MINOR: {n_long_h} long chunk(s) ({pct_long:.1f}%). '
        'Marginal impact; monitor retrieval precision for those chunks.'
    )
else:
    positives.append(f'\u2713  No unusually long chunks (> {LONG_THR} tokens).')

# 4. section coverage
if pct_with < 30 and n_chunks > 5:
    issues.append(
        f'\u26a0  LOW SECTION COVERAGE: {pct_with:.1f}% labelled. '
        'Section metadata will be sparse for retrieval filtering. '
        'Enrich source documents with Markdown headings.'
    )
elif pct_with >= 50:
    positives.append(f'\u2713  Section coverage: {pct_with:.1f}% of chunks labelled.')

# 5. stranded headings
if n_stranded > 0:
    issues.append(
        f'\u26a0  STRANDED HEADINGS: {n_stranded} chunk(s) heading-only. '
        'Try --strategy paragraph to keep headings with their content.'
    )
else:
    positives.append('\u2713  No heading\u2013content separation issues.')

# 6. token spread
cv = stats['Std Dev'] / stats['Mean'] if stats['Mean'] else 0
if cv > 0.55:
    issues.append(
        f'\u26a0  HIGH LENGTH VARIANCE: CV = {cv:.2f}. '
        'Expected for sentence strategy on heterogeneous documents; '
        'monitor dense retrieval uniformity once indexes are built.'
    )
else:
    positives.append(f'\u2713  Token-length spread acceptable (CV = {cv:.2f}).')

# ── determine verdict ─────────────────────────────────────────────
blocking = any(
    kw in i for i in issues
    for kw in ('CORPUS TOO SMALL', 'HIGH SHORT', 'HIGH LONG')
)
if not blocking:
    verdict_text = 'READY TO FREEZE \u2014 proceed to retrieval experiments'
    verdict_sym  = '\u2705'
else:
    verdict_text = 'DO NOT FREEZE \u2014 address blocking issues first'
    verdict_sym  = '\u274c'

# ── print ─────────────────────────────────────────────────────────
W = 64
def _box_line(s='', fill='\u2502'):
    print(f'{fill}  {s}')

print('\u250c' + '\u2500' * W + '\u2510')
_box_line('CHUNKING CONFIGURATION RECOMMENDATION')
print('\u251c' + '\u2500' * W + '\u2524')
_box_line(f'Verdict: {verdict_sym}  {verdict_text}')
print('\u251c' + '\u2500' * W + '\u2524')
_box_line('Positives:')
for p in positives:
    for line in textwrap.wrap(p, W - 2):
        _box_line('  ' + line)
if issues:
    print('\u251c' + '\u2500' * W + '\u2524')
    _box_line('Issues:')
    for iss in issues:
        for line in textwrap.wrap(iss, W - 2):
            _box_line('  ' + line)
print('\u251c' + '\u2500' * W + '\u2524')
_box_line('Configuration (as recorded in JSONL):')
_box_line(f'  strategy   : {", ".join(df["strategy"].unique())}')
_box_line(f'  corpus     : {n_docs} docs | {n_chunks} chunks | {total_tokens:,} tokens')
_box_line()
_box_line('NOTE: This notebook does NOT modify any parameters.')
_box_line('Re-run build_corpus.py manually to change chunk size / strategy.')
print('\u2514' + '\u2500' * W + '\u2518')